# Детекция меланомы | Часть 1: Борьба с дисбалансом классов

| | |
|---|---|
| **Датасет** | ISIC 2020 (33 126 изображений, менее 2% меланом) |
| **Задача** | Бинарная классификация: меланома / доброкачественное |
| **Архитектура** | ResNet-50 pretrained ImageNet, full fine-tuning |
| **Главная метрика** | AUC-PR (Average Precision) |
| **Вспомогательные** | AUC-ROC, F1, Sensitivity, Specificity, Balanced Accuracy |

---

## Логика экспериментов

Все эксперименты устроены по принципу **контролируемого изменения одного фактора** - фиксирую всё, кроме того, что изучаю. Аугментация считается частью препроцессинга и фиксируется до выбора стратегии балансировки, чтобы стратегии сравнивались в одинаковых условиях.

| Эксперимент | Что фиксируется | Что варьируется |
|---|---|---|
| **1. Сплит** | нет аугментации, обычный батч, BCE | размер val/test (10%, 12.5%, 15%) |
| **2. Аугментация** | лучший сплит, обычный батч, BCE | 5 стратегий аугментации |
| **3. Балансировка** | лучший сплит + лучшая аугментация | 5 стратегий борьбы с дисбалансом |


## 1. Конфигурация и импорты

In [1]:
import os
import copy
import json
import base64
import numpy as np
import pandas as pd
import random
from PIL import Image
from IPython.display import display, HTML
import base64

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, f1_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # не засоряю логи лишним

In [35]:
# Пути к датасетам на Kaggle
DATA_DIR  = "/kaggle/input/competitions/siim-isic-melanoma-classification"
IMAGE_DIR = "/kaggle/input/datasets/cdeotte/jpeg-melanoma-256x256/train"
CSV_PATH  = os.path.join(DATA_DIR, "train.csv")

# Гиперпараметры - всё в одном месте для воспроизводимости
IMAGE_SIZE  = 224   # стандартный входной размер для ResNet/ViT/Swin
BATCH_SIZE  = 32    # размер батча
NUM_EPOCHS  = 10    # число эпох обучения

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")

Устройство: cuda


In [36]:
# Воспроизводимость: seed=42 везде
SEED = 42 
random.seed(SEED); np.random.seed(SEED) # фиксация для Python, NumPy
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED) # фиксация для инициализации весов на CPU, GPU
torch.backends.cudnn.deterministic = True # фиксация для детерменирвоанных алгоритмов свёртки
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED) # фиксация хэширования 

## 2. Загрузка данных

Датасет ISIC 2020. Целевой столбец `target`: 0 = доброкачественное, 1 = меланома.


In [37]:
df = pd.read_csv(CSV_PATH)
print(df.head())
print(f"Всего записей: {len(df)}")

     image_name  patient_id     sex  age_approx anatom_site_general_challenge  \
0  ISIC_2637011  IP_7279968    male        45.0                     head/neck   
1  ISIC_0015719  IP_3075186  female        45.0               upper extremity   
2  ISIC_0052212  IP_2842074  female        50.0               lower extremity   
3  ISIC_0068279  IP_6890425  female        45.0                     head/neck   
4  ISIC_0074268  IP_8723313  female        55.0               upper extremity   

  diagnosis benign_malignant  target  
0   unknown           benign       0  
1   unknown           benign       0  
2     nevus           benign       0  
3   unknown           benign       0  
4   unknown           benign       0  
Всего записей: 33126


In [38]:
counts = df["target"].value_counts()
total  = len(df)
print(f"Доля меланом: {counts[1] / total:.2%} ({counts[1]} из {total})")
print(f"Соотношение классов: 1 : {counts[0] // counts[1]}")

Доля меланом: 1.76% (584 из 33126)
Соотношение классов: 1 : 55


Вывод: менее 2% - это меланомы, то есть Accuracy бесполезна, так как модель, всегда предсказывающая 0, даёт свыше 98% точности. Главная метрика - AUC-PR: честна при дисбалансе, не зависит от порога

## 3. Препроцессинг и класс датасета


In [39]:
# Базовый препроцессинг БЕЗ аугментации.
base_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),       # меньшую сторону сжимаем до 224 пикселей
    transforms.CenterCrop(IMAGE_SIZE),   # берём центральный квадрат 224×224
    transforms.ToTensor(),               # получаем тензор с дробными числами
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],      # ResNet обучался на ImageNet, где все пиксели были отмасштабированы определённым образом
        std=[0.229, 0.224, 0.225]        # Чтобы веса модели корректно работали с новыми изображениям, их необходимо соответствующе нормализовать
    ),
])

In [40]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_name"] + ".jpg")
        image    = Image.open(img_path).convert("RGB")  # защита от RGBA/grayscale
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row["target"], dtype=torch.float32)  # float32 для BCEWithLogitsLoss
        return image, label

# Быстрая проверка
_ds = MelanomaDataset(df.head(5), IMAGE_DIR, transform=base_transforms)
img, lbl = _ds[0]
print(f"Форма тензора: {img.shape}")
print(f"Диапазон значений: [{img.min():.2f}, {img.max():.2f}]")
del _ds

Форма тензора: torch.Size([3, 224, 224])
Диапазон значений: [-1.06, 2.45]


## 4. Вспомогательные функции

In [41]:
def make_split_loaders(df, test_size, train_transform=None, random_state=SEED):
    """
    Разбивает df на train/val/test. test_size=0.10 -> 10% test, 10% val, 80% train.
    Стратификация по target: доля меланом одинакова в каждой части.
    Возвращает: regular_loader, balanced_loader, val_loader, test_loader.
    """
    if train_transform is None:
        train_transform = base_transforms

    df_tv, df_test = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df["target"]
    )
    val_size = test_size / (1 - test_size)
    df_train, df_val = train_test_split(
        df_tv, test_size=val_size, random_state=random_state, stratify=df_tv["target"]
    )

    total = len(df)
    print(f"Сплит {int(test_size*100)}/{int(test_size*100)}: "
          f"train {len(df_train)} ({len(df_train)/total:.1%}), "
          f"val {len(df_val)} ({len(df_val)/total:.1%}), "
          f"test {len(df_test)} ({len(df_test)/total:.1%})")

    train_ds = MelanomaDataset(df_train, IMAGE_DIR, transform=train_transform)
    val_ds   = MelanomaDataset(df_val,   IMAGE_DIR, transform=base_transforms)  # val всегда без аугм.
    test_ds  = MelanomaDataset(df_test,  IMAGE_DIR, transform=base_transforms)  # test всегда без аугм.

    # Обычный загрузчик: случайный порядок батчей
    regular_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=4, persistent_workers=True, pin_memory=True)

    # Сбалансированный загрузчик: меланомы и здоровые тянутся с равной вероятностью
    labels = train_ds.df["target"].values
    n_neg, n_pos = (labels == 0).sum(), (labels == 1).sum()
    sample_weights = torch.tensor(
        [1.0/n_pos if l == 1 else 1.0/n_neg for l in labels], dtype=torch.float32
    )
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    balanced_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                 shuffle=False, num_workers=4,
                                 persistent_workers=True, pin_memory=True)

    val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, persistent_workers=True, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, persistent_workers=True, pin_memory=True)

    return regular_loader, balanced_loader, val_loader, test_loader, df_train

In [42]:
def make_resnet50():
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(DEVICE)

## 5. Функции обучения и оценки

**Ключевое методологическое решение про порог (threshold):**  
Порог - это гиперпараметр, так что подбирать его на тестовых данных - это data leakage.  
Ищу оптимальный порог на **валидации** и применяем **тот же** на тесте.


In [43]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Одна эпоха обучения. Возвращает средний лосс за эпоху."""
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)  # [B] -> [B,1] для BCEWithLogitsLoss
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [44]:
def compute_metrics(model, loader, threshold=None):
    """
    Считает все метрики модели на данных.

    threshold=None  -> ищет оптимальный порог по F1 на валидации
    threshold=число -> применяет этот порог без поиска на тесте
    """
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(model(images.to(DEVICE))).squeeze(1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    
    # Защита от NaN: если модель выдала нечисловые значения - trial провальный
    if np.isnan(all_probs).any() or np.isinf(all_probs).any():
        all_probs = np.nan_to_num(all_probs, nan=0.0, posinf=1.0, neginf=0.0)

    # AUC-PR: площадь под Precision-Recall кривой.
    # Главная метрика при дисбалансе: не зависит от порога, штрафует за каждую пропущенную меланому. 
    # AUC-ROC при дисбалансе обманчив - модель, всегда предсказывающая 0, даёт высокий AUC-ROC.
    auc_pr = average_precision_score(all_labels, all_probs)

    # Подбор оптимального порога по F1
    # Порог - граница решения: если вероятность >= порога -> предсказываем меланому. По умолчанию порог = 0,5, но при дисбалансе лучший порог часто намного ниже.
    # Перебираю все пороги от 0,01 до 0,99 с шагом 0,01, беру лучший по F1.
    if threshold is None:
        best_f1, best_thr = 0.0, 0.5
        for thr in np.arange(0.01, 0.99, 0.01):
            preds = (all_probs >= thr).astype(int)
            f1 = f1_score(all_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, float(thr)
    else:
        best_thr = float(threshold)
        preds    = (all_probs >= best_thr).astype(int)
        best_f1  = f1_score(all_labels, preds, zero_division=0)

    preds = (all_probs >= best_thr).astype(int)

    # Матрица ошибок — четыре возможных исхода:
    # TP (True Positive):  предсказали меланому — она есть  -> правильно, поймали
    # TN (True Negative):  предсказали норму    — она есть  -> правильно, не напугали
    # FP (False Positive): предсказали меланому — её нет    -> ложная тревога
    # FN (False Negative): предсказали норму    — меланома  -> пропустили болезнь
    tp = int(((preds == 1) & (all_labels == 1)).sum())
    tn = int(((preds == 0) & (all_labels == 0)).sum())
    fp = int(((preds == 1) & (all_labels == 0)).sum())
    fn = int(((preds == 0) & (all_labels == 1)).sum())

    # Recall (или же Sensitivity в медицине) = TP / (TP + FN)
    # "Из всех реальных меланом — сколько мы поймали?"
    # Самый важный вопрос медицины: пропущенная меланома опаснее ложной тревоги.
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Precision = TP / (TP + FP)
    # "Из всех кого мы назвали меланомой - сколько ею реально оказались?"
    # Важна для пациента: не хочется зря пугаться.
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # Specificity = TN / (TN + FP)
    # "Из всех здоровых - скольких мы правильно назвали здоровыми?"
    # Проверяю и метрику на доброкачественных образованиях
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # F1 = (2 * Precision * Recall) / (Precision + Recall)
    # Среднее гармоническое Precision и Recall.
    # Хорош когда нужен баланс: не только ловить меланомы, но и не кричать зря.
    # Уже посчитан выше при подборе порога, переиспользуем.

    # Balanced Accuracy = (Recall + Specificity) / 2
    # Среднее качество на обоих классах одновременно.
    balanced_acc = (recall + specificity) / 2.0

    return {
        "auc_pr":       auc_pr,
        "f1":           best_f1,
        "recall":       recall,       # = Sensitivity: сколько меланом поймали
        "precision":    precision,    # из предсказанных меланом — сколько верных
        "specificity":  specificity,  # сколько здоровых правильно определили
        "balanced_acc": balanced_acc, # честный средний балл по обоим классам
        "threshold":    best_thr,     # порог, при котором посчитаны пороговые метрики
    }

In [45]:
def run_experiment(name, train_loader, val_loader, test_loader, criterion=None):
    """
    Полный цикл обучения + оценки одного эксперимента.
    1. Одинаковый старт для всех моделей — make_resnet50() внутри
    2. Обучаем NUM_EPOCHS эпох, после каждой считаем val AUC-PR
    3. Итоговая val-метрика — среднее AUC-PR за последние 3 эпохи (стабильнее одного пика)
    4. Порог подбирается на val по финальной модели — применяется на тесте без утечки данных
    """
    if criterion is None:
        criterion = nn.BCEWithLogitsLoss()
    print()
    print(f"--- Обучение: {name} ---")
    model     = make_resnet50()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    val_auc_pr_history = []  # запоминаю val AUC-PR каждой эпохи

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss  = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = compute_metrics(model, val_loader, threshold=None)
        val_auc_pr_history.append(val_metrics["auc_pr"])

        # Среднее за последние 3 эпохи
        last3_mean = np.mean(val_auc_pr_history[-3:])

        print(
            f"  Эп {epoch:02d}/{NUM_EPOCHS} | loss {train_loss:.4f} | "
            f"val AUC-PR {val_metrics['auc_pr']:.4f} | "
            f"last3 mean {last3_mean:.4f} | F1 {val_metrics['f1']:.4f}",
            flush=True
        )

    # Итоговая метрика - среднее за последние 3 эпохи
    final_val_auc_pr = np.mean(val_auc_pr_history[-3:])

    # Порог считаю на финальной модели по val заново
    final_val_metrics = compute_metrics(model, val_loader, threshold=None)
    final_threshold   = final_val_metrics["threshold"]

    # Применяю порог на test
    test_m = compute_metrics(model, test_loader, threshold=final_threshold)
    print(
        f"  Val AUC-PR (mean last 3): {final_val_auc_pr:.4f} | "
        f"TEST: AUC-PR {test_m['auc_pr']:.4f} | "
        f"F1 {test_m['f1']:.4f} (thr={test_m['threshold']:.2f}) | "
        f"Recall {test_m['recall']:.4f} | Precision {test_m['precision']:.4f} | "
        f"Spec {test_m['specificity']:.4f} | BalAcc {test_m['balanced_acc']:.4f}"
    )
    return {
        "name":           name,
        "val_auc_pr":     final_val_auc_pr,
        "test_auc_pr":    test_m["auc_pr"],
        "test_f1":        test_m["f1"],
        "test_recall":    test_m["recall"],
        "test_precision": test_m["precision"],
        "test_spec":      test_m["specificity"],
        "test_bal_acc":   test_m["balanced_acc"],
        "test_thr":       test_m["threshold"],
        "model":          model,
    }

In [46]:
def print_results_table(results, title):
    """Таблица результатов экспериментов."""
    W = 122
    print()
    print("=" * W)
    print(f"  {title}")
    print("=" * W)
    print(f"{'Эксперимент':<38} {'Val AUC-PR':>10} {'AUC-PR':>8} {'F1':>7} "
          f"{'Recall':>8} {'Precision':>10} {'Spec':>7} {'BalAcc':>8}")
    print("-" * W)
    for r in results:
        print(f"{r['name']:<38} {r['val_auc_pr']:>10.4f} {r['test_auc_pr']:>8.4f} {r['test_f1']:>7.4f} "
              f"{r['test_recall']:>8.4f} {r['test_precision']:>10.4f} "
              f"{r['test_spec']:>7.4f} {r['test_bal_acc']:>8.4f}")
    best = max(results, key=lambda x: x["val_auc_pr"])  # именно по валидации
    print()
    print(f"Лучший по Val AUC-PR: {best['name']} -> val {best['val_auc_pr']:.4f} | test {best['test_auc_pr']:.4f}")
    return best

## 6. Эксперимент 1: выбор сплита

**Что делаю:** перебираем три варианта разбиения данных.

**Что фиксирую:** обычный батч + BCE + без аугментации - нет ничего лишнего, чистый baseline.

**Что варьирую:** доля val/test (10%, 12.5%, 15%).

Слишком маленький test -> мало меланом (58) -> нестабильная оценка AUC-PR.  
Слишком большой test -> мало train -> модель хуже обучается.  
Задача: найти баланс.

In [53]:
split_results = []
for split_size in [0.10, 0.125, 0.15]:
    regular_loader, _, val_loader, test_loader, _ = make_split_loaders(df, split_size)
    result = run_experiment(
        name=f"Сплит {int(split_size*100)}/{int(split_size*100)}",
        train_loader=regular_loader,
        val_loader=val_loader,
        test_loader=test_loader
    )
    result["split_size"] = split_size
    split_results.append(result)

best_split = print_results_table(split_results, "Эксперимент 1: выбор сплита")
best_size  = best_split["split_size"]
print(f"Лучший сплит: {int(best_size*100)}/{int(best_size*100)} - использую далее.")

Сплит 10/10: train 26500 (80.0%), val 3313 (10.0%), test 3313 (10.0%)

--- Обучение: Сплит 10/10 ---
  Эп 01/10 | loss 0.0836 | val AUC-PR 0.0782 | last3 mean 0.0782 | F1 0.1493
  Эп 02/10 | loss 0.0712 | val AUC-PR 0.1000 | last3 mean 0.0891 | F1 0.2171
  Эп 03/10 | loss 0.0681 | val AUC-PR 0.1489 | last3 mean 0.1091 | F1 0.2308
  Эп 04/10 | loss 0.0620 | val AUC-PR 0.1095 | last3 mean 0.1195 | F1 0.2345
  Эп 05/10 | loss 0.0532 | val AUC-PR 0.1093 | last3 mean 0.1226 | F1 0.1806
  Эп 06/10 | loss 0.0361 | val AUC-PR 0.0788 | last3 mean 0.0992 | F1 0.1694
  Эп 07/10 | loss 0.0177 | val AUC-PR 0.0704 | last3 mean 0.0861 | F1 0.0976
  Эп 08/10 | loss 0.0119 | val AUC-PR 0.1026 | last3 mean 0.0839 | F1 0.1556
  Эп 09/10 | loss 0.0124 | val AUC-PR 0.0529 | last3 mean 0.0753 | F1 0.0971
  Эп 10/10 | loss 0.0049 | val AUC-PR 0.0545 | last3 mean 0.0700 | F1 0.1168
  Val AUC-PR (mean last 3): 0.0700 | TEST: AUC-PR 0.1431 | F1 0.2388 (thr=0.01) | Recall 0.2759 | Precision 0.2105 | Spec 0.9816 

### Вывод по Эксперименту 1

**Наблюдаемая проблема: переобучение во всех трёх вариантах.** Начиная примерно с 5-6 эпохи обучающий лосс резко падает (у сплита 10/10 — с 0.0532 на 5-й эпохе до 0.0049 на 10-й), тогда как val AUC-PR перестаёт расти и начинает колебаться. Это классический признак того, что модель запоминает обучающую выборку вместо того, чтобы обобщаться. Данная проблема характерна для сильно несбалансированных датасетов: модель быстро выучивает доминирующий класс (здоровые пациенты) и теряет способность обобщаться на меланомы.

**Сплит 10/10** показал наименьший val AUC-PR (0.0700), несмотря на то что train-выборка здесь самая большая (26 500 изображений). Причина: при малом val (3 313 изображений, из которых меланом около 66) оценка нестабильна: небольшое число положительных примеров делает AUC-PR чувствительным к случайным флуктуациям. Переобучение здесь выражено наиболее ярко.

**Сплит 15/15** дал val AUC-PR 0.0877 - лучше чем 10/10, но хуже 12.5/12.5. При увеличении val и test до 15% train-выборка сокращается до 23 188 изображений, что негативно сказывается на качестве обучения при и без того сложном дисбалансе классов.

**Сплит 12.5/12.5** показал наилучший val AUC-PR (0.1196) и наилучший test AUC-PR (0.1348). Это разбиение обеспечивает баланс между достаточным объёмом обучающей выборки (24 844 изображения) и репрезентативностью val/test (по 4 141 изображению, 83 меланомы в каждой части). Val AUC-PR при этом более стабилен по эпохам по сравнению с остальными вариантами.

**Вывод:** для дальнейших экспериментов выбран сплит **12.5/12.5/75**. Разбиение зафиксировано в CSV-файлах для обеспечения честной воспроизводимости всех последующих сравнений.

## 7. Эксперимент 2: выбор стратегии аугментации

**Почему аугментация до стратегий балансировки?**  
В литературе по медицинскому CV и в пайплайнах ISIC аугментация рассматривается как **препроцессинг**, то есть часть подготовки данных, которая фиксируется до выбора стратегии обучения. Это позволяет затем сравнивать стратегии балансировки в одинаковых условиях.

**Что фиксируется:** лучший сплит + обычный батч + BCE.

**Что варьируется:** 5 стратегий аугментации для train.

Сравниваю с базой: лучший сплит + BCE + **без аугментации**.

In [14]:
# Определяю 5 стратегий аугментации
augmentation_strategies = {

    "Базовая геометрическая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),   # случайный вырез вместо центрального
        transforms.RandomHorizontalFlip(),   # отражение по горизонтали (p=0,5)
        transforms.RandomVerticalFlip(),     # отражение по вертикали (p=0,5)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Расширенная геометрическая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),  # поворот в диапазоне 45 градусов
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Цветовая аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        # ColorJitter - случайные вариации яркости/контраста/насыщенности
        # Важно для дерматоскопии: освещение и аппарат влияют на цвет снимка
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Комбинированная аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),

    "Агрессивная аугментация": transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=45),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        # RandomErasing: закрашивает случайный прямоугольник
        # Симулирует артефакты дерматоскопии: волосы, блики, пузырьки геля
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
    ]),
}

In [17]:
# Базовая точка: лучший сплит + без аугментации + BCE (результат из эксперимента 1).
aug_results = [{
    "name":           f"Без аугментации (Сплит {int(best_size*100)}/{int(best_size*100)})",
    "val_auc_pr":     best_split["val_auc_pr"],
    "test_auc_pr":    best_split["test_auc_pr"],
    "test_f1":        best_split["test_f1"],
    "test_recall":    best_split["test_recall"],
    "test_precision": best_split["test_precision"],
    "test_spec":      best_split["test_spec"],
    "test_bal_acc":   best_split["test_bal_acc"],
    "test_thr":       best_split["test_thr"],
}]

# Запускаю 5 аугментаций с обычным батчем и BCE
for aug_name, aug_transform in augmentation_strategies.items():
    regular_loader, _, val_loader, test_loader, _ = make_split_loaders(
        df, best_size, train_transform=aug_transform
    )
    result = run_experiment(
        name=aug_name,
        train_loader=regular_loader,
        val_loader=val_loader,
        test_loader=test_loader
    )
    aug_results.append(result)

best_aug      = print_results_table(aug_results, "Эксперимент 2: выбор аугментации")
best_aug_name = best_aug["name"]

# Если лучшей оказалась базовая точка без аугментации - берём None, make_split_loaders при train_transform=None сам подставит base_transforms
best_aug_transform = augmentation_strategies.get(best_aug_name, None)
print(f"Лучшая аугментация: {best_aug_name}.")

Сплит 12/12: train 24844 (75.0%), val 4141 (12.5%), test 4141 (12.5%)

--- Обучение: Базовая геометрическая аугментация ---
  Эп 01/10 | loss 0.0876 | val AUC-PR 0.0935 | last3 mean 0.0935 | F1 0.1958
  Эп 02/10 | loss 0.0745 | val AUC-PR 0.1433 | last3 mean 0.1184 | F1 0.1887
  Эп 03/10 | loss 0.0735 | val AUC-PR 0.1755 | last3 mean 0.1375 | F1 0.2424
  Эп 04/10 | loss 0.0703 | val AUC-PR 0.1336 | last3 mean 0.1508 | F1 0.2289
  Эп 05/10 | loss 0.0708 | val AUC-PR 0.1717 | last3 mean 0.1603 | F1 0.2264
  Эп 06/10 | loss 0.0687 | val AUC-PR 0.1574 | last3 mean 0.1542 | F1 0.2321
  Эп 07/10 | loss 0.0660 | val AUC-PR 0.1966 | last3 mean 0.1752 | F1 0.2446
  Эп 08/10 | loss 0.0664 | val AUC-PR 0.1912 | last3 mean 0.1817 | F1 0.2836
  Эп 09/10 | loss 0.0656 | val AUC-PR 0.1769 | last3 mean 0.1882 | F1 0.2687
  Эп 10/10 | loss 0.0642 | val AUC-PR 0.1675 | last3 mean 0.1785 | F1 0.2254
  Val AUC-PR (mean last 3): 0.1785 | TEST: AUC-PR 0.1238 | F1 0.1818 (thr=0.15) | Recall 0.2877 | Precisio

### Вывод по Эксперименту 2

**Базовая геометрическая аугментация** (RandomCrop + горизонтальное и вертикальное отражение) показала наилучший результат - val AUC-PR **0.1785**, что на 0.056 выше базовой точки без аугментации. Обучение при этом стабильно: лосс снижается плавно, val AUC-PR растёт на протяжении большинства эпох без резких провалов. Горизонтальное и вертикальное отражение особенно оправданы в дерматоскопии: меланома не имеет анатомически фиксированной ориентации, и модель не должна опираться на пространственное положение образования.

**Расширенная геометрическая аугментация** (добавлен RandomRotation до 45 градусов) дала val AUC-PR 0.1642 - хуже базовой геометрической. При этом test AUC-PR оказался выше (0.1917 против 0.1238), однако выбор по тестовой метрике недопустим: тест не участвует в принятии решений, иначе это утечка данных. Вероятная причина отставания по val: поворот на 45 градусов создаёт угловые артефакты при кадрировании (чёрные треугольники по краям), которые модель вынуждена игнорировать, что усложняет обучение без пропорциональной пользы.

**Цветовая аугментация** (ColorJitter без геометрических преобразований) показала наихудший результат среди всех стратегий — val AUC-PR 0.0693. Поведение лосса характерно для переобучения: резкое падение с 5-й эпохи (0.0596 -> 0.0101) при одновременном ухудшении val AUC-PR. Отсутствие геометрических преобразований означает, что каждое изображение всегда подаётся в одной и той же пространственной конфигурации - модель быстро запоминает расположение признаков, а не их суть. Цветовые вариации при этом не компенсируют эту проблему.

**Комбинированная аугментация** (геометрия + поворот + ColorJitter) дала val AUC-PR 0.1452 - лучше цветовой и агрессивной, но хуже базовой геометрической. Заметно, что лосс снижается значительно медленнее (0.0692 на 10-й эпохе против 0.0642 у базовой), что говорит о том, что модель получает более сложные и разнообразные примеры, однако за 10 эпох не успевает извлечь из них максимум пользы.

**Агрессивная аугментация** (всё вышеперечисленное + RandomErasing) показала val AUC-PR 0.1345. RandomErasing симулирует реальные артефакты дерматоскопии - волосы, блики, пузырьки геля - и теоретически должен повышать robustness модели. Однако при небольшом числе эпох и без балансировки классов дополнительная сложность скорее мешает: модель не успевает сойтись, val AUC-PR нестабилен на протяжении всех эпох.

**Общее наблюдение:** стратегии с сильной аугментацией (комбинированная, агрессивная) потенциально полезны при более длительном обучении, однако в условиях данного эксперимента (10 эпох, без балансировки) они уступают более простому подходу. Это согласуется с литературой: избыточная аугментация при дисбалансе классов может усиливать нестабильность обучения, поскольку и без того редкие меланомы становятся ещё труднее для распознавания.

**Вывод:** для дальнейших экспериментов выбрана **базовая геометрическая аугментация**. Она обеспечивает наилучший val AUC-PR при стабильном обучении и методологически обоснована для задачи детекции меланомы.

## 8. Эксперимент 3: стратегии борьбы с дисбалансом классов

**Что фиксируется:** лучший сплит + лучшая аугментация.  
**Что варьируется:** 5 стратегий борьбы с дисбалансом.

Базовая точка сравнения - результат эксперимент 2 (лучший сплит + лучшая аугментация + обычный батч + BCE). Это честный baseline: аугментация та же, меняется только стратегию балансировки.

| Стратегия | Идея | Риск |
|---|---|---|
| **Оверсэмплинг** | физически дублируются меланомы 1:1 | переобучение под одни примеры |
| **Weighted BCE** | штраф за ошибку на меланоме х55 | гиперпараметр pos_weight нужно считать |
| **Balanced batch** | сэмплер вытаскивает классы поровну | меланомы всё равно повторяются |
| **Focal Loss** | автоподавление лёгких примеров | чувствителен к гиперпараметрам (alpha/gamma) |
| **Balanced batch + Focal Loss** | двойная атака на дисбаланс | сложнее отлаживать |


In [15]:
def oversample_train(df_train, random_state=SEED):
    """
    Физически дублирует меланомы до соотношения 1:1 со здоровыми.
    Риск: одни и те же 584 меланомы видятся примерно 50 раз за эпоху -> возможно переобучение.
    """
    df_neg    = df_train[df_train["target"] == 0]
    df_pos    = df_train[df_train["target"] == 1]
    df_pos_up = df_pos.sample(len(df_neg), replace=True, random_state=random_state)
    df_out = (
        pd.concat([df_neg, df_pos_up])
        .sample(frac=1, random_state=random_state)  # перемешиваю - меланомы не одним блоком
        .reset_index(drop=True)
    )
    print(f"Оверсэмплинг: {len(df_neg)} здоровых + {len(df_pos_up)} меланом = {len(df_out)}")
    return df_out

In [16]:
class FocalLoss(nn.Module):
    """
    Focal Loss решает проблему BCE при дисбалансе: умножает лосс каждого примера на (1-p)^gamma.
    Лёгкие примеры (высокая p) -> малый вес -> почти не учимся.
    Сложные (меланомы, малая p) -> вес близок к 1 -> активно учимся.

    alpha - дополнительный вес для меланом.
    gamma - сила подавления лёгких примеров (0 = обычный BCE).
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce          = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt           = torch.exp(-bce)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

In [17]:
def make_focal_objective(train_loader, val_loader):
    """Создаёт функцию для Optuna, которая:
      1. Получает от Optuna предложенные alpha и gamma
      2. Обучает модель с этими параметрами
      3. Возвращает val AUC-PR - Optuna запомнит результат и предложит
         лучшие параметры в следующей попытке
    Почему функция внутри функции?
    Optuna требует, чтобы её функция принимала только trial.
    Но ещё нужны train_loader и val_loader — они передаются снаружи, чтобы они были видны внутренней функции."""
    def objective(trial):
        # Optuna предлагает значения из заданных диапазонов
        alpha = trial.suggest_float("alpha", *ALPHA_RANGE)
        gamma = trial.suggest_float("gamma", *GAMMA_RANGE)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
        model     = make_resnet50()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        recent_aucs = []

        for epoch in range(1, OPTUNA_EPOCHS + 1):
            model.train()
            for images, labels in train_loader:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE).unsqueeze(1)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                # При экстремальных значениях гиперпараметров (например, alpha=0.49, gamma=0.66) Focal Loss вызывал взрыв градиентов: обучение прерывалось ошибкой Input contains NaN
                # Gradient clipping ограничивает норму градиентов до 1, направление обучения не меняется, только размер шага
                # Намеренно не использую функцию train_one_epoch, чтобы не модифицировать её для всех экспериментов, так как тогда будет нечестное сравнение остальных экспериментов
                # Gradient clipping нужен только при подборе гиперпараметров Focal Loss, где экстремальные значения alpha и gamma могут вызвать взрыв градиентов
                # Остальные стратегии были численно стабильны
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            val_metrics     = compute_metrics(model, val_loader, threshold=None)
            val_auc_pr = val_metrics["auc_pr"]
            recent_aucs.append(val_auc_pr)
            # Сообщаю Optuna о результате этой эпохи
            # Pruner сравнивает его с медианой других trials на той же эпохе
            trial.report(val_auc_pr, epoch)
            if trial.should_prune():
                # Trial явно хуже большинства - обрываю, не трачу GPU
                raise optuna.exceptions.TrialPruned()

        return float(np.mean(recent_aucs[-3:]))
    return objective


def run_optuna_focal(name, train_loader, val_loader, test_loader):
    """
    Запускает Optuna для подбора alpha/gamma, затем обучает финальную модель с лучшими параметрами и возвращает результат в стандартном формате.
    """
    print(f"Optuna: поиск гиперпараметров для '{name}' ({N_TRIALS} trials)...")

    # MedianPruner: не прунит первые N_STARTUP_TRIALS trials (слишком мало данных для медианы) и первые PRUNER_WARMUP эпох (модель ещё не разогрелась)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=PRUNER_WARMUP
    )
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(
        make_focal_objective(train_loader, val_loader),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    best_alpha = study.best_params["alpha"]
    best_gamma = study.best_params["gamma"]
    print(f"  Лучшие параметры: alpha={best_alpha:.4f}, gamma={best_gamma:.4f}")
    print(f"  Лучший val AUC-PR в поиске: {study.best_value:.4f}")
    pruned  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"  Pruned trials: {pruned}/{N_TRIALS}")

    print(f"  Финальное обучение с alpha={best_alpha:.4f}, gamma={best_gamma:.4f}...")
    result = run_experiment(
        name, train_loader, val_loader, test_loader,
        criterion=FocalLoss(alpha=best_alpha, gamma=best_gamma)
    )

    result["optuna_alpha"] = best_alpha
    result["optuna_gamma"] = best_gamma
    return result

In [18]:
# Сохраняю результаты сюда, чтобы при сбросе сессии в Kaggle не потерять результаты
RESULTS_PATH = "/kaggle/working/exp3_results.json"

In [19]:
# Текущая лучшая модель
BEST_SIZE = 0.125   # лучший сплит из эксперимента 1
BEST_AUG_NAME = "Базовая геометрическая аугментация"  # лучшая аугментация из эксперимента 2

In [20]:
best_aug_transform = augmentation_strategies.get(BEST_AUG_NAME, None)
print(f"Используем аугментацию: {BEST_AUG_NAME}")
print(f"Используем сплит: {int(BEST_SIZE*100)}/{int(BEST_SIZE*100)}")

Используем аугментацию: Базовая геометрическая аугментация
Используем сплит: 12/12


In [21]:
# Optuna: параметры поиска
# Диапазоны alpha и gamma стандартны для Focal Loss в медицинском CV.
N_TRIALS      = 10
ALPHA_RANGE   = (0.1, 0.5)   # вес для меланом
GAMMA_RANGE   = (0.5, 5.0)   # сила подавления лёгких примеров
OPTUNA_EPOCHS = 5            # эпох на один trial во время поиска (экономия GPU), финальное обучение всё равно NUM_EPOCHS=10 эпох
PRUNER_WARMUP = 3            # не прунить первые 3 эпохи - модели нужно время разогреться

In [22]:
def save_results(results):
    """
    Сохраняет все результаты в JSON и выводит кликабельную ссылку для скачивания.
    """
    serializable = [{k: v for k, v in r.items() if k != "model"} for r in results]
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(serializable, f, ensure_ascii=False, indent=2)

    # Создаём inline ссылку для скачивания прямо из ячейки (Kaggle любит не сохранять результаты)
    with open(RESULTS_PATH, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(RESULTS_PATH)
    href = (
        f'<a href="data:application/json;base64,{b64}" download="{filename}" '
        f'style="font-size:14px; font-weight:bold;">'
        f'Скачать {filename} ({len(serializable)} стратегий)</a>'
    )
    display(HTML(href))
    print(f"Сохранено в {RESULTS_PATH}")

In [23]:
# Загрузчики с лучшей аугментацией
regular_loader, balanced_loader, val_loader, test_loader, df_train_s = make_split_loaders(
    df, BEST_SIZE, train_transform=best_aug_transform
)

# pos_weight для Weighted BCE: кол-во здоровых / кол-во меланом = 55
# Смысл: ошибка на одной меланоме = ошибка на 55 здоровых
n_neg      = (df_train_s["target"] == 0).sum()
n_pos      = (df_train_s["target"] == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"pos_weight = {pos_weight.item():.1f}x (train: {n_neg} здоровых, {n_pos} меланом)")

Сплит 12/12: train 24844 (75.0%), val 4141 (12.5%), test 4141 (12.5%)
pos_weight = 55.7x (train: 24406 здоровых, 438 меланом)


In [24]:
strategy_results = []

In [28]:
# Стратегия 1: Оверсэмплинг
df_train_over = oversample_train(df_train_s)
over_ds       = MelanomaDataset(df_train_over, IMAGE_DIR, transform=best_aug_transform)
over_loader   = DataLoader(over_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, persistent_workers=True, pin_memory=True)
result = run_experiment("Оверсэмплинг", over_loader, val_loader, test_loader)
strategy_results.append(result)
save_results(strategy_results)

Оверсэмплинг: 24406 здоровых + 24406 меланом = 48812

--- Обучение: Оверсэмплинг ---
  Эп 01/10 | loss 0.2191 | val AUC-PR 0.1576 | last3 mean 0.1576 | F1 0.2685
  Эп 02/10 | loss 0.0998 | val AUC-PR 0.1755 | last3 mean 0.1665 | F1 0.2716
  Эп 03/10 | loss 0.0638 | val AUC-PR 0.2190 | last3 mean 0.1840 | F1 0.3043
  Эп 04/10 | loss 0.0491 | val AUC-PR 0.1433 | last3 mean 0.1793 | F1 0.2378
  Эп 05/10 | loss 0.0389 | val AUC-PR 0.1796 | last3 mean 0.1806 | F1 0.2479
  Эп 06/10 | loss 0.0352 | val AUC-PR 0.1861 | last3 mean 0.1697 | F1 0.2523
  Эп 07/10 | loss 0.0299 | val AUC-PR 0.1358 | last3 mean 0.1672 | F1 0.2082
  Эп 08/10 | loss 0.0298 | val AUC-PR 0.1930 | last3 mean 0.1716 | F1 0.2635
  Эп 09/10 | loss 0.0258 | val AUC-PR 0.1092 | last3 mean 0.1460 | F1 0.1818
  Эп 10/10 | loss 0.0244 | val AUC-PR 0.1983 | last3 mean 0.1668 | F1 0.2464
  Val AUC-PR (mean last 3): 0.1668 | TEST: AUC-PR 0.1893 | F1 0.2603 (thr=0.59) | Recall 0.2603 | Precision 0.2603 | Spec 0.9867 | BalAcc 0.6235


Сохранено в /kaggle/working/exp3_results.json


In [29]:
# Стратегия 2: Weighted BCE
result = run_experiment(
    "Weighted BCE", regular_loader, val_loader, test_loader,
    criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
)
strategy_results.append(result)
save_results(strategy_results)


--- Обучение: Weighted BCE ---
  Эп 01/10 | loss 1.0743 | val AUC-PR 0.1495 | last3 mean 0.1495 | F1 0.2236
  Эп 02/10 | loss 0.9705 | val AUC-PR 0.1166 | last3 mean 0.1331 | F1 0.2292
  Эп 03/10 | loss 0.9385 | val AUC-PR 0.1026 | last3 mean 0.1229 | F1 0.1972
  Эп 04/10 | loss 0.8966 | val AUC-PR 0.1415 | last3 mean 0.1202 | F1 0.1609
  Эп 05/10 | loss 0.8598 | val AUC-PR 0.1575 | last3 mean 0.1339 | F1 0.2149
  Эп 06/10 | loss 0.8314 | val AUC-PR 0.1229 | last3 mean 0.1406 | F1 0.2069
  Эп 07/10 | loss 0.8628 | val AUC-PR 0.1809 | last3 mean 0.1537 | F1 0.2332
  Эп 08/10 | loss 0.8007 | val AUC-PR 0.1178 | last3 mean 0.1405 | F1 0.2254
  Эп 09/10 | loss 0.8192 | val AUC-PR 0.1110 | last3 mean 0.1366 | F1 0.2115
  Эп 10/10 | loss 0.8100 | val AUC-PR 0.1213 | last3 mean 0.1167 | F1 0.2162
  Val AUC-PR (mean last 3): 0.1167 | TEST: AUC-PR 0.1406 | F1 0.1711 (thr=0.96) | Recall 0.2192 | Precision 0.1404 | Spec 0.9759 | BalAcc 0.5975


Сохранено в /kaggle/working/exp3_results.json


In [30]:
# Стратегия 3: Balanced batch
result = run_experiment("Balanced batch", balanced_loader, val_loader, test_loader)
strategy_results.append(result)
save_results(strategy_results)


--- Обучение: Balanced batch ---
  Эп 01/10 | loss 0.2771 | val AUC-PR 0.2100 | last3 mean 0.2100 | F1 0.2817
  Эп 02/10 | loss 0.1498 | val AUC-PR 0.1577 | last3 mean 0.1839 | F1 0.2314
  Эп 03/10 | loss 0.1046 | val AUC-PR 0.2011 | last3 mean 0.1896 | F1 0.2941
  Эп 04/10 | loss 0.0771 | val AUC-PR 0.1643 | last3 mean 0.1744 | F1 0.2233
  Эп 05/10 | loss 0.0675 | val AUC-PR 0.2321 | last3 mean 0.1992 | F1 0.2895
  Эп 06/10 | loss 0.0590 | val AUC-PR 0.1650 | last3 mean 0.1871 | F1 0.2687
  Эп 07/10 | loss 0.0531 | val AUC-PR 0.2311 | last3 mean 0.2094 | F1 0.3040
  Эп 08/10 | loss 0.0447 | val AUC-PR 0.2104 | last3 mean 0.2021 | F1 0.2750
  Эп 09/10 | loss 0.0429 | val AUC-PR 0.2197 | last3 mean 0.2204 | F1 0.2615
  Эп 10/10 | loss 0.0370 | val AUC-PR 0.1521 | last3 mean 0.1941 | F1 0.2086
  Val AUC-PR (mean last 3): 0.1941 | TEST: AUC-PR 0.2302 | F1 0.3214 (thr=0.70) | Recall 0.3699 | Precision 0.2842 | Spec 0.9833 | BalAcc 0.6766


Сохранено в /kaggle/working/exp3_results.json


Focal Loss чувствителен к гиперпараметрам alpha и gamma - запускать с дефолтными значениями было бы нечестно по отношению к другим стратегиям. Использую Optuna для поиска лучшей комбинации гиперпараметров.

**Optuna TPE sampler** - байесовская оптимизация: учится на предыдущих trials и быстрее находит хорошую область, в отличие от случайного перебора.

**MedianPruner** - обрывает trial после PRUNER_WARMUP эпох, если его промежуточный результат хуже медианы всех остальных trials на той же эпохе. Это экономит GPU-время, не теряя качества поиска.

In [25]:
# Стратегия 4: Focal Loss
result = run_optuna_focal("Focal Loss", regular_loader, val_loader, test_loader)
strategy_results.append(result)
save_results(strategy_results)

Optuna: поиск гиперпараметров для 'Focal Loss' (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
  8%|▊         | 7.75M/97.8M [00:00<00:01, 80.9MB/s]
 31%|███▏      | 30.6M/97.8M [00:00<00:00, 174MB/s] 
 55%|█████▍    | 53.4M/97.8M [00:00<00:00, 203MB/s]
 78%|███████▊  | 75.9M/97.8M [00:00<00:00, 216MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


  Лучшие параметры: alpha=0.2226, gamma=0.9834
  Лучший val AUC-PR в поиске: 0.1798
  Pruned trials: 4/10
  Финальное обучение с alpha=0.2226, gamma=0.9834...

--- Обучение: Focal Loss ---
  Эп 01/10 | loss 0.0096 | val AUC-PR 0.1202 | last3 mean 0.1202 | F1 0.2074
  Эп 02/10 | loss 0.0087 | val AUC-PR 0.1146 | last3 mean 0.1174 | F1 0.2014
  Эп 03/10 | loss 0.0086 | val AUC-PR 0.2055 | last3 mean 0.1468 | F1 0.2500
  Эп 04/10 | loss 0.0083 | val AUC-PR 0.1236 | last3 mean 0.1479 | F1 0.2143
  Эп 05/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.1156 | F1 0.0000
  Эп 06/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.0529 | F1 0.0000
  Эп 07/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.0176 | F1 0.0000
  Эп 08/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.0176 | F1 0.0000
  Эп 09/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.0176 | F1 0.0000
  Эп 10/10 | loss nan | val AUC-PR 0.0176 | last3 mean 0.0176 | F1 0.0000
  Val AUC-PR (mean last 3): 0.0176 | TEST: AUC-PR 0.0176 | 

Сохранено в /kaggle/working/exp3_results.json


In [25]:
# Стратегия 5: Balanced batch + Focal Loss
result = run_optuna_focal("Balanced batch + Focal Loss", balanced_loader, val_loader, test_loader)
strategy_results.append(result)
save_results(strategy_results)

Optuna: поиск гиперпараметров для 'Balanced batch + Focal Loss' (10 trials)...


  0%|          | 0/10 [00:00<?, ?it/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 10%|█         | 9.88M/97.8M [00:00<00:00, 104MB/s]
 30%|██▉       | 28.9M/97.8M [00:00<00:00, 160MB/s]
 50%|████▉     | 48.9M/97.8M [00:00<00:00, 182MB/s]
 73%|███████▎  | 71.4M/97.8M [00:00<00:00, 203MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]


  Лучшие параметры: alpha=0.4806, gamma=1.5417
  Лучший val AUC-PR в поиске: 0.2076
  Pruned trials: 1/10
  Финальное обучение с alpha=0.4806, gamma=1.5417...

--- Обучение: Balanced batch + Focal Loss ---
  Эп 01/10 | loss 0.0488 | val AUC-PR 0.1627 | last3 mean 0.1627 | F1 0.2235
  Эп 02/10 | loss 0.0285 | val AUC-PR 0.1977 | last3 mean 0.1802 | F1 0.2680
  Эп 03/10 | loss 0.0208 | val AUC-PR 0.1168 | last3 mean 0.1591 | F1 0.2059
  Эп 04/10 | loss 0.0162 | val AUC-PR 0.1761 | last3 mean 0.1635 | F1 0.2840
  Эп 05/10 | loss 0.0140 | val AUC-PR 0.1757 | last3 mean 0.1562 | F1 0.2639
  Эп 06/10 | loss 0.0110 | val AUC-PR 0.1682 | last3 mean 0.1734 | F1 0.2809
  Эп 07/10 | loss 0.0107 | val AUC-PR 0.1483 | last3 mean 0.1641 | F1 0.2190
  Эп 08/10 | loss 0.0091 | val AUC-PR 0.1645 | last3 mean 0.1604 | F1 0.2581
  Эп 09/10 | loss 0.0084 | val AUC-PR 0.1495 | last3 mean 0.1541 | F1 0.1974
  Эп 10/10 | loss 0.0079 | val AUC-PR 0.2113 | last3 mean 0.1751 | F1 0.3046
  Val AUC-PR (mean last 

Сохранено в /kaggle/working/exp3_results.json


In [27]:
BASELINE_METRICS = {
    "name": "Baseline",
    "val_auc_pr": 0.1785,
    "test_auc_pr": 0.1238,
    "test_f1": 0.1818,
    "test_recall": 0.2877,
    "test_precision": 0.1329,
    "test_spec": 0.9663,
    "test_bal_acc": 0.6270,
    "test_thr": 0.15,
}

In [28]:
# Загружаю каждый файл отдельно, так как обучение долгое, сессию приходилось обрывать каждые 12 часов
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

results_1 = load_json("/kaggle/input/datasets/nikitaliya/part1-disbalance-nir/exp3_results (1).json")
results_2 = load_json("/kaggle/input/datasets/nikitaliya/part1-disbalance-nir/exp3_results (2).json")
results_3 = load_json("/kaggle/input/datasets/nikitaliya/part1-disbalance-nir/exp3_results (3).json")

all_results = [BASELINE_METRICS] + results_1 + results_2 + results_3

seen = set()
strategy_results = []
for r in all_results:
    if r["name"] not in seen:
        seen.add(r["name"])
        strategy_results.append(r)

In [31]:
df = pd.DataFrame([{
    "Стратегия":    r["name"],
    "val AUC-PR":   r["val_auc_pr"],
    "test AUC-PR":  r["test_auc_pr"],
    "test F1":      r["test_f1"],
    "test Recall":  r["test_recall"],
    "test Prec.":   r["test_precision"],
    "test Spec.":   r["test_spec"],
    "test BalAcc":  r["test_bal_acc"],
    "thr":          r["test_thr"],
} for r in strategy_results])

numeric_cols = df.columns.drop("Стратегия")

def highlight_best(col):
    if col.name == "Стратегия":
        return [""] * len(col)
    best = col.max()
    return [
        "background-color: #d4edda; font-weight: bold" if v == best else ""
        for v in col
    ]

df.style \
    .apply(highlight_best) \
    .format({c: "{:.4f}" for c in df.columns if c not in ("Стратегия", "thr")} | {"thr": "{:.2f}"}) \
    .hide(axis="index") \
    .set_caption("Сравнение стратегий борьбы с дисбалансом классов")

Стратегия,val AUC-PR,test AUC-PR,test F1,test Recall,test Prec.,test Spec.,test BalAcc,thr
Baseline,0.1785,0.1238,0.1818,0.2877,0.1329,0.9663,0.6270,0.15
Оверсэмплинг,0.1668,0.1893,0.2603,0.2603,0.2603,0.9867,0.6235,0.59
Weighted BCE,0.1167,0.1406,0.1711,0.2192,0.1404,0.9759,0.5975,0.96
Balanced batch,0.1941,0.2302,0.3214,0.3699,0.2842,0.9833,0.6766,0.70
Focal Loss,0.0176,0.0176,0.0000,0.0000,0.0000,1.0000,0.5000,0.50
Balanced batch + Focal Loss,0.1751,0.1677,0.2484,0.2740,0.2273,0.9833,0.6286,0.53


### Вывод по Эксперименту 3

**Balanced batch** показал наилучший результат - val AUC-PR **0.1941**, test AUC-PR **0.2302**, что заметно выше baseline по обеим метрикам. Recall составил 0.3699 - модель улавливает больше меланом, чем любая другая стратегия, при этом сохраняя приемлемую Specificity (0.9833) и Balanced Accuracy (0.6766). Механизм WeightedRandomSampler вытягивает меланомы и здоровые примеры с равной вероятностью в каждом батче, не дублируя физически изображения, что снижает риск переобучения по сравнению с оверсэмплингом.

**Оверсэмплинг** дал val AUC-PR 0.1668 - ниже baseline, хотя test AUC-PR (0.1893) выглядит неплохо. Однако выбор производится по валидационной метрике, и здесь оверсэмплинг уступает. Физическое дублирование 584 меланом до соотношения 1:1 означает, что модель видит одни и те же примеры десятки раз за эпоху, что ведёт к переобучению на конкретные изображения меланом, а не к обобщению признаков.

**Weighted BCE** (pos_weight = 55) показал val AUC-PR 0.1167. Лосс при этом принципиально другого масштаба - около 0.8–1.0 против 0.06–0.08 у остальных стратегий, что отражает сильный штраф за каждую ошибку на меланоме. Модель в итоге выдаёт завышенные вероятности для всех примеров, и оптимальный порог сдвигается до 0.96 - то есть модель уверенно называет меланомой лишь те случаи, в которых абсолютно уверена. Это объясняет низкий Recall (0.2192): большинство меланом не преодолевают столь высокую планку. Стратегия оказалась наименее эффективной среди рассмотренных.

**Focal Loss** (подбор alpha и gamma через Optuna) показал наихудший результат - val AUC-PR 0.0176. Несмотря на то что в процессе поиска Optuna нашла конфигурацию с val AUC-PR 0.1798, финальное обучение с найденными параметрами оказалось нестабильным: стратегия не подошла для данной задачи в рамках выбранных условий эксперимента.

**Balanced batch + Focal Loss** (подбор alpha и gamma через Optuna) дал val AUC-PR 0.1751 - ниже чем просто Balanced batch. Комбинация двух механизмов балансировки не дала синергии в рамках данного эксперимента.

**Вывод:** для дальнейших экспериментов выбрана стратегия **Balanced batch** (WeightedRandomSampler). Она обеспечивает наилучший val AUC-PR (0.1941) и наилучший test AUC-PR (0.2302) при стабильном обучении, высоком Recall и без риска переобучения, характерного для физического оверсэмплинга.

In [48]:
# Воссоздаю сплиты с теми же параметрами, что использовались при обучении
df_tv, df_test = train_test_split(df, test_size=BEST_SIZE, random_state=SEED, stratify=df["target"])
df_train, df_val = train_test_split(df_tv, test_size=BEST_SIZE/(1-BEST_SIZE), random_state=SEED, stratify=df_tv["target"])

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 24844, Val: 4141, Test: 4141


In [49]:
# Сохраняю только image_name и target - этого достаточно, чтобы воссоздать датасет
df_train[["image_name", "target"]].to_csv("/kaggle/working/split_train.csv", index=False)
df_val[["image_name", "target"]].to_csv("/kaggle/working/split_val.csv", index=False)
df_test[["image_name", "target"]].to_csv("/kaggle/working/split_test.csv", index=False)

In [50]:
# Функция для генерации ссылки на скачивание прямо из ноутбука: читаем файл, переводим в base64 и встраиваем в HTML-ссылку
def make_download_link(filepath, filename):
    with open(filepath, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()
    html = f'<a href="data:file/csv;base64,{encoded}" download="{filename}">{filename}</a>'
    display(HTML(html))

In [51]:
make_download_link("/kaggle/working/split_train.csv", "split_train.csv")
make_download_link("/kaggle/working/split_val.csv", "split_val.csv")
make_download_link("/kaggle/working/split_test.csv", "split_test.csv")